In [1]:
import os
import sys
import json

try:
    import google.colab
    from google.colab import drive
    from google.colab import userdata
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("We are in Colab!")
    drive.mount('/content/drive', force_remount=True)
    
    !pip install -q tqdm requests openpyxl pandas scikit-learn
    
    ROOT_DIR = "/content/drive/MyDrive/omg_diploma_2025/restore_punct"
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    try:
        API_KEY = userdata.get('YANDEX_API_KEY')
        FOLDER_ID = userdata.get('YANDEX_FOLDER_ID')
    except userdata.SecretNotFoundError:
        raise ValueError("Please set YANDEX_API_KEY and YANDEX_FOLDER_ID in Colab Secrets!")

else:
    print("We are running locally!")
    ROOT_DIR = os.getcwd()
    if ROOT_DIR not in sys.path:
        sys.path.append(ROOT_DIR)
        
    secrets_path = os.path.join(ROOT_DIR, "yandex_secrets.json")
    if os.path.exists(secrets_path):
        with open(secrets_path, "r", encoding="utf-8") as f:
            secrets = json.load(f)
            API_KEY = secrets.get("YANDEX_API_KEY")
            FOLDER_ID = secrets.get("YANDEX_FOLDER_ID")
            if not API_KEY or not FOLDER_ID:
                raise ValueError("YANDEX_API_KEY or YANDEX_FOLDER_ID is missing in yandex_secrets.json")
    else:
        raise FileNotFoundError(f"Please create {secrets_path} with your API keys!")

DATA_DIR = os.path.join(ROOT_DIR, "data")
RESULTS_DIR = os.path.join(ROOT_DIR, "results")
YANDEX_DB_PATH = os.path.join(RESULTS_DIR, "yandex_db.json")
YANDEX_EXCEL_PATH = os.path.join(RESULTS_DIR, "yandex_results_table.xlsx")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


We are running locally!


In [2]:
import time
import requests
import datetime
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

try:
    from extract_labels_standardized import extract_labels_standardized, ID_TO_LABEL
except ImportError:
    raise ImportError("Error: Could not import extract_labels_standardized. Make sure the file exists in your ROOT_DIR.")

EXPERIMENT_NAME = "YandexGPT_Pro_FewShot_v2"

TEST_DATASETS = {
    "General_Test": os.path.join(DATA_DIR, "bench_test_all.json"),
    "GERA_Test": os.path.join(DATA_DIR, "bench_test_gera.json"),
    "LORuGEC_Test": os.path.join(DATA_DIR, "bench_test_lorugec.json"),
}


In [3]:
def prompt_yandex_to_restore(unpunctuated_text):
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {API_KEY}",
        "x-folder-id": FOLDER_ID
    }

    system_prompt = (
        "Ты — профессиональный редактор русского языка. "
        "Твоя задача — расставить знаки препинания в тексте, где они полностью удалены. "
        "Строго соблюдай правила:\n"
        "1. Ни в коем случае не изменяй, не удаляй и не добавляй новые слова. "
        "Не исправляй регистр букв. Только вставляй пунктуацию.\n"
        "2. Не объединяй и не разделяй слова через дефис.\n"
        "3. Выведи только восстановленный текст, без каких-либо комментариев и кавычек."
    )

    body = {
        "modelUri": f"gpt://{FOLDER_ID}/yandexgpt/latest",
        "completionOptions": {
            "stream": False,
            "temperature": 0.0,
            "maxTokens": "4000"
        },
        "messages": [
            {"role": "system", "text": system_prompt},
            {"role": "user", "text": "Мама мыла раму а папа читал газету"},
            {"role": "assistant", "text": "Мама мыла раму, а папа читал газету."},
            {"role": "user", "text": "Он сказал что придет завтра а она ответила Хорошо"},
            {"role": "assistant", "text": 'Он сказал, что придет завтра, а она ответила: "Хорошо".'},
            {"role": "user", "text": unpunctuated_text}
        ]
    }

    try:
        response = requests.post(url, headers=headers, json=body)
        if response.status_code == 200:
            result = response.json()
            return result['result']['alternatives'][0]['message']['text']
        else:
            print(f"API Error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"Request failed: {e}")
        return None


In [ ]:
def _align_by_words(true_tokens, llm_pairs):
    """Align LLM predictions to gold tokens by matching words, not positions.

    Returns a list of predicted label ids, one per gold token.
    Handles cases where YandexGPT merges words (e.g. "ИТ специалистов" ->
    "ИТ-специалистов") or adds/drops tokens by scanning forward.
    """
    pred_labels = []
    pred_idx = 0
    pred_words = [p["word"].lower() for p in llm_pairs]

    for gold_word in true_tokens:
        gold_lower = gold_word.lower()
        if pred_idx < len(pred_words) and pred_words[pred_idx] == gold_lower:
            pred_labels.append(llm_pairs[pred_idx]["label"])
            pred_idx += 1
        elif pred_idx < len(pred_words) and gold_lower in pred_words[pred_idx]:
            pred_labels.append(llm_pairs[pred_idx]["label"])
        else:
            found = False
            for ahead in range(1, min(4, len(pred_words) - pred_idx)):
                if pred_words[pred_idx + ahead] == gold_lower:
                    pred_idx += ahead
                    pred_labels.append(llm_pairs[pred_idx]["label"])
                    pred_idx += 1
                    found = True
                    break
            if not found:
                pred_labels.append(0)  # fallback: O

    return pred_labels


def evaluate_yandex_on_dataset(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    all_true_tags = []
    all_pred_tags = []
    skipped = 0

    for idx, item in enumerate(tqdm(dataset, desc=f"Evaluating {os.path.basename(json_path)}")):
        true_tokens = item["tokens"]
        true_labels = item["ner_tags"]

        unpunctuated_text = " ".join(true_tokens)
        llm_output_text = prompt_yandex_to_restore(unpunctuated_text)

        if llm_output_text is None:
            skipped += 1
            continue

        if idx < 3:
            print(f"\n[Preview {idx+1} from {os.path.basename(json_path)}]")
            print(f"--- sent text: {unpunctuated_text[:200]}")
            print(f"--- received:  {llm_output_text[:200]}")

        llm_pairs = extract_labels_standardized(llm_output_text)
        aligned_pred = _align_by_words(true_tokens, llm_pairs)

        for t, p in zip(true_labels, aligned_pred):
            if t != -100:
                all_true_tags.append(ID_TO_LABEL.get(t, "O"))
                all_pred_tags.append(ID_TO_LABEL.get(p, "O"))

        time.sleep(0.6)

    if skipped:
        print(f"  ⚠ skipped {skipped}/{len(dataset)} items due to API errors")

    return classification_report(all_true_tags, all_pred_tags, output_dict=True, zero_division=0)


current_experiment_results = {"timestamp": str(datetime.datetime.now())}

for test_name, file_path in TEST_DATASETS.items():
    print(f"\nStarting {test_name} with {EXPERIMENT_NAME}...")

    report = evaluate_yandex_on_dataset(file_path)

    current_experiment_results[test_name] = report
    wa = report.get('weighted avg', {})
    ma = report.get('macro avg', {})
    print(f"\nFinished {test_name}.")
    print(f"  W-F1={wa.get('f1-score',0):.4f}  M-F1={ma.get('f1-score',0):.4f}")



Starting General_Test with YandexGPT_Pro_FewShot_v2...


Evaluating bench_test_all.json:   0%|          | 0/569 [00:00<?, ?it/s]


[Preview 1 from bench_test_all.json]
--- sent text: В советский период времени число ИТ специалистов в Армении составляло около десяти тысяч Доставшийся в наследство от советского периода времени промышленный и интеллектуальный потенциал оказался благо
--- received:  В советский период времени число ИТ-специалистов в Армении составляло около десяти тысяч. Доставшийся в наследство от советского периода времени промышленный и интеллектуальный потенциал оказался благ

[Preview 2 from bench_test_all.json]
--- sent text: Примерно столько же детей обучалось и к 1991 году в 1307 начальной и средней школах С 2001 02 учебного года согласно Государственной программе развития образования в Армении действует 11-летняя систем
--- received:  Примерно столько же детей обучалось и к 1991 году в 1307 начальной и средней школах. С 2001/02 учебного года, согласно Государственной программе развития образования в Армении, действует 11-летняя сис

[Preview 3 from bench_test_all.json]
--- sent text: В Армен

Evaluating bench_test_gera.json:   0%|          | 0/1259 [00:00<?, ?it/s]


[Preview 1 from bench_test_gera.json]
--- sent text: Как показана правда времени в романе А С Пушкина Капитанская дочка
--- received:  Как показана правда времени в романе А. С. Пушкина «Капитанская дочка»?

[Preview 2 from bench_test_gera.json]
--- sent text: Действия в романе происходят в золотом веке в момент правления Екатерины II
--- received:  Действия в романе происходят в золотом веке, в момент правления Екатерины II.

[Preview 3 from bench_test_gera.json]
--- sent text: Конечно были исторические произведения и до этого
--- received:  Конечно, были исторические произведения и до этого.

Finished GERA_Test.
  W-F1=0.9570  M-F1=0.4118

Starting LORuGEC_Test with YandexGPT_Pro_FewShot_v2...


Evaluating bench_test_lorugec.json:   0%|          | 0/603 [00:00<?, ?it/s]


[Preview 1 from bench_test_lorugec.json]
--- sent text: Вася попробовал и так и эдак но у него все равно ничего не вышло
--- received:  Вася попробовал и так и эдак, но у него всё равно ничего не вышло.

[Preview 2 from bench_test_lorugec.json]
--- sent text: Кругом лес ни конца ни края
--- received:  Кругом лес, ни конца ни края.

[Preview 3 from bench_test_lorugec.json]
--- sent text: От него целый год не было ни слуху ни духу
--- received:  От него целый год не было ни слуху ни духу.

Finished LORuGEC_Test.
  W-F1=0.9651  M-F1=0.4117


In [ ]:
if os.path.exists(YANDEX_DB_PATH):
    with open(YANDEX_DB_PATH, "r", encoding="utf-8") as f:
        yandex_db = json.load(f)
else:
    yandex_db = {}

yandex_db[EXPERIMENT_NAME] = current_experiment_results

with open(YANDEX_DB_PATH, "w", encoding="utf-8") as f:
    json.dump(yandex_db, f, indent=4, ensure_ascii=False)

print(f"\nResults saved to JSON: {YANDEX_DB_PATH}")

ALL_TEST_NAMES = list(TEST_DATASETS.keys())
summary_records = []
detailed_records = []

for model_name, data in yandex_db.items():
    sum_row = {"Experiment": model_name}
    for test_name in ALL_TEST_NAMES:
        if test_name in data and "weighted avg" in data[test_name]:
            wa = data[test_name]["weighted avg"]
            ma = data[test_name].get("macro avg", {})
            sum_row[(test_name, "Weighted_F1")] = round(wa["f1-score"] * 100, 2)
            sum_row[(test_name, "Weighted_P")] = round(wa["precision"] * 100, 2)
            sum_row[(test_name, "Weighted_R")] = round(wa["recall"] * 100, 2)
            sum_row[(test_name, "Macro_F1")] = round(ma.get("f1-score", 0) * 100, 2)
            sum_row[(test_name, "Accuracy")] = round(data[test_name].get("accuracy", 0) * 100, 2)
    summary_records.append(sum_row)

    labels = set()
    for test_name in ALL_TEST_NAMES:
        if test_name in data:
            labels.update(k for k in data[test_name] if k not in ("accuracy", "macro avg", "weighted avg", "timestamp"))

    for label in sorted(labels):
        det_row = {"Experiment": model_name, "Punctuation": label}
        for test_name in ALL_TEST_NAMES:
            if test_name in data and label in data[test_name]:
                metrics = data[test_name][label]
                det_row[(test_name, "F1")] = round(metrics["f1-score"] * 100, 2)
                det_row[(test_name, "Precision")] = round(metrics["precision"] * 100, 2)
                det_row[(test_name, "Recall")] = round(metrics["recall"] * 100, 2)
                det_row[(test_name, "Support")] = metrics["support"]
        detailed_records.append(det_row)

df_summary = pd.DataFrame(summary_records)
df_detailed = pd.DataFrame(detailed_records)

if not df_summary.empty:
    df_summary.set_index("Experiment", inplace=True)
    df_summary.columns = pd.MultiIndex.from_tuples(df_summary.columns)

if not df_detailed.empty:
    df_detailed.set_index(["Experiment", "Punctuation"], inplace=True)
    df_detailed.columns = pd.MultiIndex.from_tuples(df_detailed.columns)

with pd.ExcelWriter(YANDEX_EXCEL_PATH, engine='openpyxl') as writer:
    if not df_summary.empty:
        df_summary.to_excel(writer, sheet_name="Summary")
    if not df_detailed.empty:
        df_detailed.to_excel(writer, sheet_name="Per-Class Details")

print(f"Exported Yandex experiments to: {YANDEX_EXCEL_PATH}")

from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()



Results saved to JSON: /home/temari/god please no diploma/restore_punct/results/yandex_db.json
Exported Yandex experiments to: /home/temari/god please no diploma/restore_punct/results/yandex_results_table.xlsx
Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 14
  Yandex runs : 1


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'